# JAX RUN

In [ ]:
import os
import time

# Set Backend ("jax" or "torch")
backend = "jax"

import numpy as np

BATCH_SIZE = 32
NUM_BATCHES = 20
WARMUP_ITERS = 3
TOTAL_SAMPLES = BATCH_SIZE * NUM_BATCHES
MODEL_ID = "openai/clip-vit-base-patch32"

compile_status = "eager"

# =============================================================================
# JAX BACKEND -- Keras 3
# =============================================================================
if backend == "jax":
    os.environ["KERAS_BACKEND"] = "jax"

    import jax
    import keras
    import keras_hub

    dummy_pixel_values = np.random.uniform(0.0, 1.0, size=(BATCH_SIZE, 224, 224, 3)).astype("float16")
    dummy_input_ids = np.random.randint(0, 49408, size=(BATCH_SIZE, 77)).astype("int32")
    input_data = {"images": dummy_pixel_values, "token_ids": dummy_input_ids}

    model = keras_hub.models.CLIPBackbone.from_preset("clip_vit_base_patch32")

    def forward_fn(data):
        return model(data, training=False)

    forward_pass = jax.jit(forward_fn)
    compile_status = "jit"

    def call_forward():
        return forward_pass(input_data)

    def sync_gpu(output=None):
        if output is not None:
            jax.block_until_ready(output)

# =============================================================================
# TORCH BACKEND
# =============================================================================
elif backend == "torch":
    import torch
    from transformers import CLIPModel

    dummy_pixel_values = np.random.uniform(0.0, 1.0, size=(BATCH_SIZE, 3, 224, 224)).astype("float32")
    dummy_input_ids = np.random.randint(0, 49408, size=(BATCH_SIZE, 77)).astype("int64")
    dummy_attention_mask = np.ones((BATCH_SIZE, 77), dtype="int64")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()

    pixel_values = torch.from_numpy(dummy_pixel_values).to(device)
    input_ids = torch.from_numpy(dummy_input_ids).to(device)
    attention_mask = torch.from_numpy(dummy_attention_mask).to(device)

    def forward_fn():
        with torch.no_grad():
            return model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask)

    try:
        compiled_model = torch.compile(model, fullgraph=False, dynamic=False)

        def compiled_forward():
            with torch.no_grad():
                return compiled_model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask)

        _ = compiled_forward()
        call_forward = compiled_forward
        compile_status = "compiled"
    except Exception as e:
        print(f"torch.compile failed, falling back to eager: {e}")
        call_forward = forward_fn
        compile_status = "eager (compile failed)"

    def sync_gpu(output=None):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

else:
    raise ValueError(f"Unknown backend: {backend}")


# 2. Warmup passes
print("Running warmup/compilation (this may take a moment)...")
for i in range(WARMUP_ITERS):
    t0 = time.perf_counter()
    out = call_forward()
    sync_gpu(out)
    t1 = time.perf_counter()
    print(f"  warmup {i + 1}/{WARMUP_ITERS}: {(t1 - t0) * 1000:.2f} ms")

# 3. Benchmark Loop
latencies = []
start_total = time.perf_counter()

for _ in range(NUM_BATCHES):
    t0 = time.perf_counter()
    out = call_forward()
    sync_gpu(out)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000.0)

total_time = time.perf_counter() - start_total

# 4. Results
latencies_arr = np.array(latencies)
label = f"{backend.upper()} ({compile_status})"
print(f"[{label}] Throughput: {TOTAL_SAMPLES / total_time:.2f} samples/sec")
print(
    f"[{label}] Avg Batch Latency: {np.mean(latencies_arr):.2f} ms "
    f"± {np.std(latencies_arr):.2f} ms"
)
print(f"[{label}] Min/Max Batch Latency: {latencies_arr.min():.2f} / {latencies_arr.max():.2f} ms")

Running warmup/compilation (this may take a moment)...
  warmup 1/3: 21214.78 ms
  warmup 2/3: 156.29 ms
  warmup 3/3: 155.29 ms
[JAX (jit)] Throughput: 201.05 samples/sec
[JAX (jit)] Avg Batch Latency: 159.13 ms ± 1.95 ms
[JAX (jit)] Min/Max Batch Latency: 153.88 / 161.62 ms


#Pytorch Run

In [ ]:
import os
import time

backend = "torch"

import numpy as np

BATCH_SIZE = 32
NUM_BATCHES = 20
WARMUP_ITERS = 3
TOTAL_SAMPLES = BATCH_SIZE * NUM_BATCHES
MODEL_ID = "openai/clip-vit-base-patch32"
compile_status = "eager"

# =============================================================================
# JAX BACKEND -- Keras 3
# =============================================================================
if backend == "jax":
    os.environ["KERAS_BACKEND"] = "jax"

    import jax
    import keras
    import keras_hub

    dummy_pixel_values = np.random.uniform(0.0, 1.0, size=(BATCH_SIZE, 224, 224, 3)).astype("float16")
    dummy_input_ids = np.random.randint(0, 49408, size=(BATCH_SIZE, 77)).astype("int32")
    input_data = {"images": dummy_pixel_values, "token_ids": dummy_input_ids}

    model = keras_hub.models.CLIPBackbone.from_preset("clip_vit_base_patch32")

    def forward_fn(data):
        return model(data, training=False)

    forward_pass = jax.jit(forward_fn)
    compile_status = "jit"

    def call_forward():
        return forward_pass(input_data)

    def sync_gpu(output=None):
        if output is not None:
            jax.block_until_ready(output)

# =============================================================================
# TORCH BACKEND
# =============================================================================
elif backend == "torch":
    import torch
    from transformers import CLIPModel

    dummy_pixel_values = np.random.uniform(0.0, 1.0, size=(BATCH_SIZE, 3, 224, 224)).astype("float32")
    dummy_input_ids = np.random.randint(0, 49408, size=(BATCH_SIZE, 77)).astype("int64")
    dummy_attention_mask = np.ones((BATCH_SIZE, 77), dtype="int64")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()

    pixel_values = torch.from_numpy(dummy_pixel_values).to(device)
    input_ids = torch.from_numpy(dummy_input_ids).to(device)
    attention_mask = torch.from_numpy(dummy_attention_mask).to(device)

    def forward_fn():
        with torch.no_grad():
            return model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask)

    try:
        compiled_model = torch.compile(model, fullgraph=False, dynamic=False)

        def compiled_forward():
            with torch.no_grad():
                return compiled_model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask)

        _ = compiled_forward()
        call_forward = compiled_forward
        compile_status = "compiled"
    except Exception as e:
        print(f"torch.compile failed, falling back to eager: {e}")
        call_forward = forward_fn
        compile_status = "eager (compile failed)"

    def sync_gpu(output=None):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

else:
    raise ValueError(f"Unknown backend: {backend}")


# 2. Warmup passes
print("Running warmup/compilation (this may take a moment)...")
for i in range(WARMUP_ITERS):
    t0 = time.perf_counter()
    out = call_forward()
    sync_gpu(out)
    t1 = time.perf_counter()
    print(f"  warmup {i + 1}/{WARMUP_ITERS}: {(t1 - t0) * 1000:.2f} ms")

# 3. Benchmark Loop
latencies = []
start_total = time.perf_counter()

for _ in range(NUM_BATCHES):
    t0 = time.perf_counter()
    out = call_forward()
    sync_gpu(out)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000.0)

total_time = time.perf_counter() - start_total

# 4. Results
latencies_arr = np.array(latencies)
label = f"{backend.upper()} ({compile_status})"
print(f"[{label}] Throughput: {TOTAL_SAMPLES / total_time:.2f} samples/sec")
print(
    f"[{label}] Avg Batch Latency: {np.mean(latencies_arr):.2f} ms "
    f"± {np.std(latencies_arr):.2f} ms"
)
print(f"[{label}] Min/Max Batch Latency: {latencies_arr.min():.2f} / {latencies_arr.max():.2f} ms")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Running warmup/compilation (this may take a moment)...
  warmup 1/3: 337.79 ms
  warmup 2/3: 169.37 ms
  warmup 3/3: 168.72 ms
[TORCH (compiled)] Throughput: 186.51 samples/sec
[TORCH (compiled)] Avg Batch Latency: 171.52 ms ± 2.13 ms
[TORCH (compiled)] Min/Max Batch Latency: 167.89 / 176.15 ms
